# Dataset

In [2]:
!nvidia-smi | head
!lscpu | grep -E "Model|Socket"
%pip install -U polars[gpu] --extra-index-url=https://pypi.nvidia.com

Mon Apr  6 13:46:13 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P8             18W /   70W |       0MiB /  15360MiB |      0%      Default |
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
Model:                                   85
Socket(s):              

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
# common libraries
import polars as pl
import json
from pathlib import Path

In [5]:
%cd /content/drive/MyDrive/UQ/Courses/S1.1/NF_CSE_CIC_IDS2018_v3/data/ 

/content/drive/MyDrive/UQ/Courses/S1.1/NF_CSE_CIC_IDS2018_v3/data


## Dataset Verification

### Raw Intake & Data Manifest

In [6]:
DATA_CSV = "NF-CICIDS2018-v3.csv"
FEATURE_CSV="NetFlow_v3_Features.csv"

dlf = pl.scan_csv(DATA_CSV, infer_schema_length=10000)
flf = pl.scan_csv(FEATURE_CSV)

display(flf.head(10).collect(engine='gpu'))
display(dlf.head(10).collect(engine='gpu'))

Feature,Description
str,str
"""IPV4_SRC_ADDR""","""IPv4 source address"""
"""IPV4_DST_ADDR""","""IPv4 destination address"""
"""L4_SRC_PORT""","""IPv4 source port number"""
"""L4_DST_PORT""","""IPv4 destination port number"""
"""PROTOCOL""","""IP protocol identifier byte"""
"""L7_PROTO""","""Layer 7 protocol (numeric)"""
"""IN_BYTES""","""Incoming number of bytes"""
"""OUT_BYTES""","""Outgoing number of bytes"""
"""IN_PKTS""","""Incoming number of packets"""


FLOW_START_MILLISECONDS,FLOW_END_MILLISECONDS,IPV4_SRC_ADDR,L4_SRC_PORT,IPV4_DST_ADDR,L4_DST_PORT,PROTOCOL,L7_PROTO,IN_BYTES,IN_PKTS,OUT_BYTES,OUT_PKTS,TCP_FLAGS,CLIENT_TCP_FLAGS,SERVER_TCP_FLAGS,FLOW_DURATION_MILLISECONDS,DURATION_IN,DURATION_OUT,MIN_TTL,MAX_TTL,LONGEST_FLOW_PKT,SHORTEST_FLOW_PKT,MIN_IP_PKT_LEN,MAX_IP_PKT_LEN,SRC_TO_DST_SECOND_BYTES,DST_TO_SRC_SECOND_BYTES,RETRANSMITTED_IN_BYTES,RETRANSMITTED_IN_PKTS,RETRANSMITTED_OUT_BYTES,RETRANSMITTED_OUT_PKTS,SRC_TO_DST_AVG_THROUGHPUT,DST_TO_SRC_AVG_THROUGHPUT,NUM_PKTS_UP_TO_128_BYTES,NUM_PKTS_128_TO_256_BYTES,NUM_PKTS_256_TO_512_BYTES,NUM_PKTS_512_TO_1024_BYTES,NUM_PKTS_1024_TO_1514_BYTES,TCP_WIN_MAX_IN,TCP_WIN_MAX_OUT,ICMP_TYPE,ICMP_IPV4_TYPE,DNS_QUERY_ID,DNS_QUERY_TYPE,DNS_TTL_ANSWER,FTP_COMMAND_RET_CODE,SRC_TO_DST_IAT_MIN,SRC_TO_DST_IAT_MAX,SRC_TO_DST_IAT_AVG,SRC_TO_DST_IAT_STDDEV,DST_TO_SRC_IAT_MIN,DST_TO_SRC_IAT_MAX,DST_TO_SRC_IAT_AVG,DST_TO_SRC_IAT_STDDEV,Label,Attack
i64,i64,str,i64,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str
1518611287705,1518611287705,"""172.31.0.2""",53,"""172.31.66.58""",63593,17,5,156,1,0,0,0,0,0,1,0,0,0,0,156,156,0,156,0,1,0,0,0,0,1248000,0,0,1,0,0,0,0,0,0,0,45856,1,60,0,0,0,0,0,0,0,0,0,0,"""Benign"""
1518611287743,1518611290747,"""172.31.66.58""",56163,"""239.255.255.250""",1900,17,12,805,5,0,0,0,0,0,3003,3003,0,1,1,161,161,0,161,0,0,0,0,0,0,2143,0,0,5,0,0,0,0,0,0,0,0,0,0,0,0,2793,750,1182,0,0,0,0,0,"""Benign"""
1518611288143,1518611300202,"""172.31.66.46""",62388,"""239.255.255.250""",1900,17,12,1288,8,0,0,0,0,0,12058,12058,0,1,1,161,161,0,161,0,0,0,0,0,0,854,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,3016,1722,1436,0,0,0,0,0,"""Benign"""
1518611288165,1518611336194,"""0.0.0.0""",546,"""0.0.0.0""",547,17,103,393,3,0,0,0,0,0,48029,48029,0,1,1,131,131,0,131,0,0,0,0,0,0,65,0,0,3,0,0,0,0,0,0,0,0,0,0,0,16014,32014,24014,8000,0,0,0,0,0,"""Benign"""
1518611288175,1518611288176,"""172.31.66.46""",49187,"""169.254.169.254""",80,6,7,373,5,700,5,27,27,27,1,0,0,128,128,528,40,40,528,700,373,0,0,0,0,1492000,2800000,8,1,0,1,0,8192,17922,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""Benign"""
1518611288214,1518611289754,"""107.217.94.48""",62907,"""172.31.66.46""",3389,6,91,1000,5,421,4,28,28,24,1539,1539,1197,106,106,621,40,40,621,0,0,0,0,0,0,5194,2187,6,2,0,1,0,252,63618,0,0,0,0,0,0,54,1252,384,501,128,937,398,380,0,"""Benign"""
1518611288295,1518611288496,"""172.31.66.46""",60810,"""172.31.0.2""",53,17,5,63,1,156,1,0,0,0,201,0,0,0,0,156,63,63,156,0,0,0,0,0,0,2495,6178,1,1,0,0,0,0,0,0,0,59209,1,60,0,0,0,0,0,0,0,0,0,0,"""Benign"""
1518611288383,1518611360054,"""172.31.66.58""",138,"""172.31.66.127""",138,17,10,2155,10,0,0,0,0,0,71670,71670,0,128,128,232,202,0,232,0,0,0,0,0,0,240,0,0,10,0,0,0,0,0,0,0,0,0,0,0,0,46728,7962,14318,0,0,0,0,0,"""Benign"""
1518611288390,1518611361578,"""172.31.66.46""",138,"""172.31.66.127""",138,17,10,2357,11,0,0,0,0,0,73188,73188,0,128,128,232,202,0,232,0,0,0,0,0,0,257,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,59450,7318,17484,0,0,0,0,0,"""Benign"""


Checking dataset for differences and chronological validity

In [7]:
feature_names = (
    flf
    .select(
        pl.col("Feature")
        .cast(pl.Utf8)
        .str.strip_chars()
        .str.to_lowercase()
        .alias("feature")
    )
    .drop_nulls()
    .unique()
    .collect()
)

header_check = (
    pl.DataFrame({
        "feature": [c.strip().lower() for c in dlf.collect_schema().names()]
    })
    .with_columns(
        pl.col("feature").is_in(feature_names["feature"].implode()).alias("in_flf")
    )
)

display(
    pl.DataFrame({
        "status": ["in_flf", "missing"],
        "count": [
            header_check.filter(pl.col("in_flf")).height,
            header_check.filter(~pl.col("in_flf")).height,
        ],
        "features": [
            header_check.filter(pl.col("in_flf"))["feature"].to_list(),
            header_check.filter(~pl.col("in_flf"))["feature"].to_list(),
        ],
    })
)

status,count,features
str,i64,list[str]
"""in_flf""",53,"[""flow_start_milliseconds"", ""flow_end_milliseconds"", … ""dst_to_src_iat_stddev""]"
"""missing""",2,"[""label"", ""attack""]"


In [8]:
start = pl.col("FLOW_START_MILLISECONDS").cast(pl.Int64, strict=False)
end = pl.col("FLOW_END_MILLISECONDS").cast(pl.Int64, strict=False)
duration = pl.col("FLOW_DURATION_MILLISECONDS").cast(pl.Int64, strict=False)
delta = end - start

chrono_check = (
    dlf
    .select(
        pl.len().alias("rows"),
        start.null_count().alias("start_nulls"),
        end.null_count().alias("end_nulls"),
        (start.is_null() | end.is_null()).sum().alias("rows_missing_anchor"),
        (end < start).sum().alias("end_before_start"),
        delta.min().alias("min_delta_ms"),
        delta.max().alias("max_delta_ms"),
        ((duration - delta).abs() > 1).sum().alias("duration_mismatch_rows"),
        start.min().alias("start_min_ms"),
        start.max().alias("start_max_ms"),
        end.min().alias("end_min_ms"),
        end.max().alias("end_max_ms"),
    )
    .collect(engine="streaming")
    .with_columns(
        pl.from_epoch("start_min_ms", time_unit="ms").alias("start_min"),
        pl.from_epoch("start_max_ms", time_unit="ms").alias("start_max"),
        pl.from_epoch("end_min_ms", time_unit="ms").alias("end_min"),
        pl.from_epoch("end_max_ms", time_unit="ms").alias("end_max"),
    )
)

display(chrono_check)

rows,start_nulls,end_nulls,rows_missing_anchor,end_before_start,min_delta_ms,max_delta_ms,duration_mismatch_rows,start_min_ms,start_max_ms,end_min_ms,end_max_ms,start_min,start_max,end_min,end_max
u32,u32,u32,u32,u32,i64,i64,u32,i64,i64,i64,i64,datetime[ms],datetime[ms],datetime[ms],datetime[ms]
20115529,0,0,0,0,0,120999,0,1518611287705,1520021065524,1518611287705,1520021184780,2018-02-14 12:28:07.705,2018-03-02 20:04:25.524,2018-02-14 12:28:07.705,2018-03-02 20:06:24.780


Building the manifest keyed by `day` that is derived from `FLOW_START_MILLISECONDS`,
for each day, record:
- row count
- time range
- binary label counts from `Label`
- attack-family counts from `Attack`
- check if timestamps are monotonic
- check rows need a repair by sorting on `FLOW_START_MILLISECONDS`

In [9]:
start = pl.col("FLOW_START_MILLISECONDS").cast(pl.Int64, strict=False)
day = pl.from_epoch(start, time_unit="ms").dt.date()

base = (
    dlf
    .with_columns(
        start.alias("__ts"),
        day.alias("flow_day"),
        pl.col("Label").cast(pl.Utf8),
        pl.col("Attack").cast(pl.Utf8),
    )
    .with_columns(
        (pl.col("__ts") >= pl.col("__ts").shift(1).over("flow_day"))
        .fill_null(True)
        .alias("__mono_row")
    )
)

daily_manifest = (
    base
    .group_by("flow_day")
    .agg(
        pl.len().alias("row_count"),
        pl.col("__ts").min().alias("start_ms"),
        pl.col("__ts").max().alias("end_ms"),
        (~pl.col("__mono_row")).sum().alias("out_of_order_rows"),
        pl.col("__mono_row").all().alias("timestamps_monotonic_before_sort"),
    )
    .with_columns(
        (pl.col("out_of_order_rows") > 0).alias("needs_chronology_repair"),
        pl.from_epoch("start_ms", time_unit="ms").alias("start_time"),
        pl.from_epoch("end_ms", time_unit="ms").alias("end_time"),
    )
    .sort("flow_day")
    .collect(engine="streaming")
)

daily_label_counts = (
    base
    .group_by("flow_day", "Label")
    .len()
    .sort(["flow_day", "len"], descending=[False, True])
    .collect(engine="streaming")
)

daily_attack_counts = (
    base
    .group_by("flow_day", "Attack")
    .len()
    .sort(["flow_day", "len"], descending=[False, True])
    .collect(engine="gpu")
)

display(daily_manifest)
display(daily_label_counts)
display(daily_attack_counts)

flow_day,row_count,start_ms,end_ms,out_of_order_rows,timestamps_monotonic_before_sort,needs_chronology_repair,start_time,end_time
date,u32,i64,i64,u32,bool,bool,datetime[ms],datetime[ms]
2018-02-14,2138838,1518611287705,1518652798824,0,true,false,2018-02-14 12:28:07.705,2018-02-14 23:59:58.824
2018-02-15,1665321,1518652800176,1518739146573,0,true,false,2018-02-15 00:00:00.176,2018-02-15 23:59:06.573
2018-02-16,1784129,1518784651856,1518818631966,0,true,false,2018-02-16 12:37:31.856,2018-02-16 22:03:51.966
2018-02-20,1829859,1519129685595,1519170787588,0,true,false,2018-02-20 12:28:05.595,2018-02-20 23:53:07.588
2018-02-21,2712803,1519171266903,1519257586925,0,true,false,2018-02-21 00:01:06.903,2018-02-21 23:59:46.925
…,…,…,…,…,…,…,…,…
2018-02-23,1933002,1519388107819,1519426449878,0,true,false,2018-02-23 12:15:07.819,2018-02-23 22:54:09.878
2018-02-27,828,1519734241964,1519775919372,0,true,false,2018-02-27 12:24:01.964,2018-02-27 23:58:39.372
2018-02-28,2032727,1519776775854,1519862399321,0,true,false,2018-02-28 00:12:55.854,2018-02-28 23:59:59.321


flow_day,Label,len
date,str,u32
2018-02-14,"""0""",1563644
2018-02-14,"""1""",575194
2018-02-15,"""0""",1567981
2018-02-15,"""1""",97340
2018-02-16,"""0""",1578503
…,…,…
2018-02-28,"""1""",84327
2018-03-01,"""0""",2043708
2018-03-01,"""1""",103825


flow_day,Attack,len
date,str,u32
2018-02-14,"""Benign""",1563644
2018-02-14,"""FTP-BruteForce""",386720
2018-02-14,"""SSH-Bruteforce""",188474
2018-02-15,"""Benign""",1567981
2018-02-15,"""DoS_attacks-GoldenEye""",61300
…,…,…
2018-02-28,"""Infilteration""",84327
2018-03-01,"""Benign""",2043708
2018-03-01,"""Infilteration""",103825


### Schema and Type audit

Assign a semantic type to every header
   - identity fields:
     - `IPV4_SRC_ADDR`, `IPV4_DST_ADDR`
   - High-risk shortcut fields requiring explicit justification before prediction use:
     - `L4_SRC_PORT`, `L4_DST_PORT`, `FLOW_START_MILLISECONDS`, `FLOW_END_MILLISECONDS`, `DNS_QUERY_ID`
   - Candidate predictive measurement fields:
     - duration, count, size, throughput, retransmission, TTL, and IAT features
   - Label fields:
     - `Label`, `Attack`

In [10]:
schema = dlf.collect_schema()
headers = schema.names()

identity_fields = {"IPV4_SRC_ADDR", "IPV4_DST_ADDR"}
shortcut_fields = {
    "L4_SRC_PORT", "L4_DST_PORT",
    "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS",
    "DNS_QUERY_ID",
}
label_fields = {"Label", "Attack"}
timestamp_fields = {"FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "FLOW_DURATION_MILLISECONDS"}
string_fields = identity_fields | label_fields

header_map = (
    pl.DataFrame({
        "header": headers,
        "raw_dtype": [str(schema[h]) for h in headers],
    })
    .with_columns(
        pl.when(pl.col("header").is_in(list(identity_fields)))
        .then(pl.lit("identity"))
        .when(pl.col("header").is_in(list(shortcut_fields)))
        .then(pl.lit("high_risk_shortcut"))
        .when(pl.col("header").is_in(list(label_fields)))
        .then(pl.lit("label"))
        .otherwise(pl.lit("predictive_measurement"))
        .alias("semantic_type"),

        pl.when(pl.col("header").is_in(list(timestamp_fields)))
        .then(pl.lit("Int64 milliseconds"))
        .when(pl.col("header").is_in(list(string_fields)))
        .then(pl.lit("Utf8"))
        .otherwise(pl.lit("numeric"))
        .alias("parse_as"),

        pl.when(pl.col("header").is_in(["L4_SRC_PORT", "L4_DST_PORT", "PROTOCOL", "L7_PROTO"]))
        .then(pl.lit("keep as code; later decide numeric, categorical, or both"))
        .when(pl.col("header").eq("DNS_QUERY_ID"))
        .then(pl.lit("treat as identifier-like unless justified"))
        .when(pl.col("header").is_in(list(identity_fields)))
        .then(pl.lit("use for inspection only"))
        .when(pl.col("header").is_in(list(label_fields)))
        .then(pl.lit("target only"))
        .otherwise(pl.lit("candidate input feature"))
        .alias("model_use"),
    )
    .sort("semantic_type", "header")
)

display(header_map)

header,raw_dtype,semantic_type,parse_as,model_use
str,str,str,str,str
"""DNS_QUERY_ID""","""Int64""","""high_risk_shortcut""","""numeric""","""treat as identifier-like unles…"
"""FLOW_END_MILLISECONDS""","""Int64""","""high_risk_shortcut""","""Int64 milliseconds""","""candidate input feature"""
"""FLOW_START_MILLISECONDS""","""Int64""","""high_risk_shortcut""","""Int64 milliseconds""","""candidate input feature"""
"""L4_DST_PORT""","""Int64""","""high_risk_shortcut""","""numeric""","""keep as code; later decide num…"
"""L4_SRC_PORT""","""Int64""","""high_risk_shortcut""","""numeric""","""keep as code; later decide num…"
…,…,…,…,…
"""SRC_TO_DST_IAT_STDDEV""","""Int64""","""predictive_measurement""","""numeric""","""candidate input feature"""
"""SRC_TO_DST_SECOND_BYTES""","""Int64""","""predictive_measurement""","""numeric""","""candidate input feature"""
"""TCP_FLAGS""","""Int64""","""predictive_measurement""","""numeric""","""candidate input feature"""


In [11]:
start = pl.col("FLOW_START_MILLISECONDS").cast(pl.Int64, strict=False)
end = pl.col("FLOW_END_MILLISECONDS").cast(pl.Int64, strict=False)
flow_duration = pl.col("FLOW_DURATION_MILLISECONDS").cast(pl.Int64, strict=False)
duration_in = pl.col("DURATION_IN").cast(pl.Int64, strict=False)
duration_out = pl.col("DURATION_OUT").cast(pl.Int64, strict=False)
delta = end - start

def iat_rules(prefix: str) -> list[pl.Expr]:
    iat_min = pl.col(f"{prefix}_MIN").cast(pl.Float64, strict=False)
    iat_avg = pl.col(f"{prefix}_AVG").cast(pl.Float64, strict=False)
    iat_std = pl.col(f"{prefix}_STDDEV").cast(pl.Float64, strict=False)
    iat_max = pl.col(f"{prefix}_MAX").cast(pl.Float64, strict=False)

    return [
        (iat_min < 0).sum().alias(f"{prefix.lower()}_min_negative"),
        (iat_avg < 0).sum().alias(f"{prefix.lower()}_avg_negative"),
        (iat_max < 0).sum().alias(f"{prefix.lower()}_max_negative"),
        (iat_std < 0).sum().alias(f"{prefix.lower()}_std_negative"),
        ((iat_min > iat_avg) & iat_min.is_not_null() & iat_avg.is_not_null()).sum().alias(f"{prefix.lower()}_min_gt_avg"),
        ((iat_avg > iat_max) & iat_avg.is_not_null() & iat_max.is_not_null()).sum().alias(f"{prefix.lower()}_avg_gt_max"),
        ((iat_min > iat_max) & iat_min.is_not_null() & iat_max.is_not_null()).sum().alias(f"{prefix.lower()}_min_gt_max"),
    ]

consistency = dlf.select(
    pl.len().alias("rows"),
    (end < start).sum().alias("end_before_start"),
    ((flow_duration - delta).abs() > 1).sum().alias("flow_duration_mismatch"),
    (duration_in < 0).sum().alias("duration_in_negative"),
    (duration_out < 0).sum().alias("duration_out_negative"),
    ((duration_in > flow_duration) & duration_in.is_not_null() & flow_duration.is_not_null()).sum().alias("duration_in_gt_flow_duration"),
    ((duration_out > flow_duration) & duration_out.is_not_null() & flow_duration.is_not_null()).sum().alias("duration_out_gt_flow_duration"),
    *iat_rules("SRC_TO_DST_IAT"),
    *iat_rules("DST_TO_SRC_IAT"),
).collect(engine="streaming")

display(consistency)

rows,end_before_start,flow_duration_mismatch,duration_in_negative,duration_out_negative,duration_in_gt_flow_duration,duration_out_gt_flow_duration,src_to_dst_iat_min_negative,src_to_dst_iat_avg_negative,src_to_dst_iat_max_negative,src_to_dst_iat_std_negative,src_to_dst_iat_min_gt_avg,src_to_dst_iat_avg_gt_max,src_to_dst_iat_min_gt_max,dst_to_src_iat_min_negative,dst_to_src_iat_avg_negative,dst_to_src_iat_max_negative,dst_to_src_iat_std_negative,dst_to_src_iat_min_gt_avg,dst_to_src_iat_avg_gt_max,dst_to_src_iat_min_gt_max
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
20115529,0,0,0,0,4106,0,0,0,0,0,832382,0,0,0,0,0,0,215581,0,0


### Feature analysis by Family

In [ ]:
schema = dlf.collect_schema()
headers = schema.names()

identity_cols = ["IPV4_SRC_ADDR", "IPV4_DST_ADDR"]
shortcut_cols = ["L4_SRC_PORT", "L4_DST_PORT", "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "DNS_QUERY_ID"]
timing_cols = [
    "FLOW_START_MILLISECONDS", "FLOW_END_MILLISECONDS", "FLOW_DURATION_MILLISECONDS",
    "DURATION_IN", "DURATION_OUT",
    "SRC_TO_DST_IAT_MIN", "SRC_TO_DST_IAT_MAX", "SRC_TO_DST_IAT_AVG", "SRC_TO_DST_IAT_STDDEV",
    "DST_TO_SRC_IAT_MIN", "DST_TO_SRC_IAT_MAX", "DST_TO_SRC_IAT_AVG", "DST_TO_SRC_IAT_STDDEV",
]
traffic_cols = [
    "IN_BYTES", "OUT_BYTES", "IN_PKTS", "OUT_PKTS",
    "SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "SRC_TO_DST_AVG_THROUGHPUT", "DST_TO_SRC_AVG_THROUGHPUT",
]
packet_cols = [
    "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT", "MIN_IP_PKT_LEN", "MAX_IP_PKT_LEN",
    "NUM_PKTS_UP_TO_128_BYTES", "NUM_PKTS_128_TO_256_BYTES", "NUM_PKTS_256_TO_512_BYTES",
    "NUM_PKTS_512_TO_1024_BYTES", "NUM_PKTS_1024_TO_1514_BYTES",
]
tcp_cols = [
    "TCP_FLAGS", "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
    "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT",
    "RETRANSMITTED_IN_BYTES", "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES", "RETRANSMITTED_OUT_PKTS",
]
sparse_cols = ["ICMP_TYPE", "ICMP_IPV4_TYPE", "DNS_QUERY_ID", "DNS_QUERY_TYPE", "DNS_TTL_ANSWER", "FTP_COMMAND_RET_CODE"]
string_like = {"IPV4_SRC_ADDR", "IPV4_DST_ADDR", "Label", "Attack"}

numeric_cols = [c for c in headers if c not in string_like]
numeric_set = set(numeric_cols)

def stat_alias(column, stat):
    return f"{column}__{stat}"

# Build one aggregate plan so the GPU sees a single lazy query instead of one query per column.
exprs = []
for c in headers:
    exprs.extend([
        pl.col(c).is_null().mean().alias(stat_alias(c, "missing_rate")),
        pl.col(c).n_unique().alias(stat_alias(c, "unique_count")),
    ])

    if c in string_like:
        continue

    n = pl.col(c).cast(pl.Float64, strict=False)
    exprs.extend([
        n.eq(0).mean().alias(stat_alias(c, "zero_fraction")),
        n.min().alias(stat_alias(c, "min")),
        n.median().alias(stat_alias(c, "median")),
        n.max().alias(stat_alias(c, "max")),
        n.quantile(0.95).alias(stat_alias(c, "q95")),
        n.quantile(0.99).alias(stat_alias(c, "q99")),
    ])

stats_row = dlf.select(exprs).collect(engine="gpu").row(0, named=True)

profile = pl.DataFrame(
    {
        "header": headers,
        "dtype": [str(schema[c]) for c in headers],
        "missing_rate": [stats_row[stat_alias(c, "missing_rate")] for c in headers],
        "zero_fraction": [stats_row[stat_alias(c, "zero_fraction")] if c in numeric_set else None for c in headers],
        "unique_count": [stats_row[stat_alias(c, "unique_count")] for c in headers],
        "min": [stats_row[stat_alias(c, "min")] if c in numeric_set else None for c in headers],
        "median": [stats_row[stat_alias(c, "median")] if c in numeric_set else None for c in headers],
        "max": [stats_row[stat_alias(c, "max")] if c in numeric_set else None for c in headers],
        "q95": [stats_row[stat_alias(c, "q95")] if c in numeric_set else None for c in headers],
        "q99": [stats_row[stat_alias(c, "q99")] if c in numeric_set else None for c in headers],
    }
)

display(profile)